# Refresh-Safe Monthly Process

This notebook contains the Phase 3 implementation and synthetic test evidence.

## Execution Guidance

This is a development and test notebook. Do not use Run All against populated test databases.

- Baseline setup cells create tables and original test populations once.
- Manual walkthrough cells demonstrate individual processing steps.
- Function-definition cells define the reusable refresh process.
- Test cells deliberately introduce invalid data or a temporary processing failure.
- Fresh-baseline setup requires a new, unused test database path.

For subsequent refreshes, load the required imports, configuration and function definitions, connect to the intended prepared test database, and call `run_monthly_refresh`.

The original analytical and Streamlit databases are outside this notebook’s test scope.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

test_folder = Path("../data/prototype/refresh_testing")
test_folder.mkdir(parents=True, exist_ok=True)

database_path = test_folder / "monthly_refresh_test.db"
connection = sqlite3.connect(database_path)

print(f"Connected to: {database_path.resolve()}")

In [ ]:
source_path = Path("../data/operational/source_cases.csv")

source_columns = [
    "Case ID",
    "Policy Number",
    "Client Number",
    "Snapshot Date",
    "Outstanding Amount",
    "Cancellation Status",
    "Cancelling Department",
    "Cancelling Agent",
]

baseline_source_df = pd.read_csv(
    source_path,
    dtype={
        "Case ID": "string",
        "Policy Number": "string",
        "Client Number": "string",
        "Cancelling Agent": "string",
    },
)

missing_columns = set(source_columns) - set(baseline_source_df.columns)

if missing_columns:
    raise ValueError(f"Missing source columns: {sorted(missing_columns)}")

baseline_source_df = baseline_source_df[source_columns].copy()

if baseline_source_df["Case ID"].isna().any():
    raise ValueError("Source contains missing Case IDs.")

if baseline_source_df["Case ID"].str.strip().eq("").any():
    raise ValueError("Source contains blank Case IDs.")

if baseline_source_df["Case ID"].duplicated().any():
    raise ValueError("Source contains duplicate Case IDs.")

baseline_source_df["Snapshot Date"] = pd.to_datetime(
    baseline_source_df["Snapshot Date"],
    errors="raise",
)

print("Source rows:", len(baseline_source_df))
print("Unique cases:", baseline_source_df["Case ID"].nunique())
print("Source structure checks passed.")

In [ ]:
updates_path = Path("../data/operational/agent_updates.csv")

baseline_updates_df = pd.read_csv(
    updates_path,
    dtype={"Case ID": "string"},
)

required_update_columns = [
    "Case ID",
    "Agent Working",
    "Agent Notes",
    "Case Status",
    "Final Outcome",
    "Completed By",
    "Completion Date",
    "Last Updated Date",
]

missing_columns = (
    set(required_update_columns)
    - set(baseline_updates_df.columns)
)

if missing_columns:
    raise ValueError(
        f"Missing update columns: {sorted(missing_columns)}"
    )

baseline_updates_df = baseline_updates_df[
    required_update_columns
].copy()

case_ids = baseline_updates_df["Case ID"]

if case_ids.isna().any() or case_ids.str.strip().eq("").any():
    raise ValueError("Agent updates contain missing or blank Case IDs.")

if case_ids.duplicated().any():
    raise ValueError("Agent updates contain duplicate Case IDs.")

source_ids = set(baseline_source_df["Case ID"])
update_ids = set(case_ids)

if source_ids != update_ids:
    raise ValueError(
        f"Cases missing agent records: {len(source_ids - update_ids)}; "
        f"agent records without source cases: {len(update_ids - source_ids)}"
    )

completed_mask = baseline_updates_df["Case Status"].eq("Completed")

automated_mask = (
    completed_mask
    & baseline_updates_df["Completed By"].eq("Automated Check")
)

print("Agent records:", len(baseline_updates_df))
print("Completed cases:", completed_mask.sum())
print("Automated completions:", automated_mask.sum())
print("Other completions:", (completed_mask & ~automated_mask).sum())
print("Source and agent Case IDs match.")

In [ ]:
existing_tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
      AND name IN ('source_cases', 'agent_updates');
    """,
    connection,
)

if not existing_tables.empty:
    raise ValueError(
        "Baseline tables already exist. Stop here to avoid "
        "overwriting test data."
    )

baseline_source_df.to_sql(
    "source_cases",
    connection,
    if_exists="fail",
    index=False,
)

baseline_updates_df.to_sql(
    "agent_updates",
    connection,
    if_exists="fail",
    index=False,
)

connection.execute(
    """
    CREATE UNIQUE INDEX source_case_id_unique
    ON source_cases ("Case ID");
    """
)

connection.execute(
    """
    CREATE UNIQUE INDEX agent_case_id_unique
    ON agent_updates ("Case ID");
    """
)

connection.commit()

print("Test baseline tables created.")
print("Source cases:", len(baseline_source_df))
print("Agent records:", len(baseline_updates_df))

In [ ]:
connection.execute(
    """
    CREATE TABLE baseline_agent_updates AS
    SELECT *
    FROM agent_updates;
    """
)

connection.execute(
    """
    CREATE UNIQUE INDEX baseline_agent_case_id_unique
    ON baseline_agent_updates ("Case ID");
    """
)

connection.commit()

baseline_summary = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS total_records,
        SUM(
            CASE WHEN "Case Status" = 'Completed'
            THEN 1 ELSE 0 END
        ) AS completed_cases,
        SUM(
            CASE WHEN "Case Status" = 'Completed'
              AND "Completed By" = 'Automated Check'
            THEN 1 ELSE 0 END
        ) AS automated_completions
    FROM baseline_agent_updates;
    """,
    connection,
)

baseline_summary

In [ ]:
print(
    "Earliest snapshot:",
    baseline_source_df["Snapshot Date"].min().date(),
)

latest_snapshot_date = baseline_source_df["Snapshot Date"].max()

next_snapshot_date = latest_snapshot_date + pd.offsets.MonthEnd(1)

print("Latest snapshot:", latest_snapshot_date.date())
print("Next test snapshot:", next_snapshot_date.date())

In [ ]:
test_population = baseline_source_df.merge(
    baseline_updates_df[
        ["Case ID", "Case Status", "Completed By"]
    ],
    on="Case ID",
    validate="one_to_one",
)

unresolved_cases = (
    test_population.loc[
        test_population["Case Status"].ne("Completed")
    ]
    .sort_values("Case ID")
    .head(8)
)

automated_case = (
    test_population.loc[
        test_population["Case Status"].eq("Completed")
        & test_population["Completed By"].eq("Automated Check")
    ]
    .sort_values("Case ID")
    .head(1)
)

other_completed_case = (
    test_population.loc[
        test_population["Case Status"].eq("Completed")
        & test_population["Completed By"].ne("Automated Check")
    ]
    .sort_values("Case ID")
    .head(1)
)

selected_existing_cases = pd.concat(
    [unresolved_cases, automated_case, other_completed_case],
    ignore_index=True,
)

if len(selected_existing_cases) != 10:
    raise ValueError("Could not select the expected 10 test cases.")

selected_existing_cases[
    ["Case ID", "Case Status", "Completed By"]
]

In [ ]:
july_snapshot_df = selected_existing_cases[
    source_columns
].copy()

july_snapshot_df["Snapshot Date"] = next_snapshot_date

changed_case_id = "DH000001"

amount_mask = july_snapshot_df["Case ID"].eq(changed_case_id)

previous_amount = july_snapshot_df.loc[
    amount_mask, "Outstanding Amount"
].iloc[0]

july_snapshot_df.loc[amount_mask, "Outstanding Amount"] = round(
    float(previous_amount) + 1.00,
    2,
)

print("Existing cases in July snapshot:", len(july_snapshot_df))
print("Snapshot date:", next_snapshot_date.date())
print("Changed case:", changed_case_id)
print("Previous amount:", previous_amount)
print(
    "New amount:",
    july_snapshot_df.loc[amount_mask, "Outstanding Amount"].iloc[0],
)

In [ ]:
new_cases_df = july_snapshot_df.iloc[:2].copy()

new_cases_df["Case ID"] = ["TEST-JUL-001", "TEST-JUL-002"]
new_cases_df["Policy Number"] = ["TEST-POL-001", "TEST-POL-002"]
new_cases_df["Client Number"] = ["TEST-CLI-001", "TEST-CLI-002"]
new_cases_df["Outstanding Amount"] = [25.00, 48.50]

existing_case_ids = set(baseline_source_df["Case ID"])

if new_cases_df["Case ID"].isin(existing_case_ids).any():
    raise ValueError("A new test Case ID already exists.")

july_snapshot_df = pd.concat(
    [july_snapshot_df, new_cases_df],
    ignore_index=True,
)

if july_snapshot_df["Case ID"].duplicated().any():
    raise ValueError("July snapshot contains duplicate Case IDs.")

print("July snapshot rows:", len(july_snapshot_df))
print(
    "Existing cases:",
    july_snapshot_df["Case ID"].isin(existing_case_ids).sum(),
)
print(
    "New cases:",
    (~july_snapshot_df["Case ID"].isin(existing_case_ids)).sum(),
)

In [ ]:
july_snapshot_path = test_folder / "snapshot_2026_07_valid.csv"

if july_snapshot_path.exists():
    raise FileExistsError(
        "The July test file already exists. "
        "Stop here to avoid overwriting it."
    )

july_snapshot_df.to_csv(
    july_snapshot_path,
    index=False,
    date_format="%Y-%m-%d",
)

print("Saved:", july_snapshot_path.resolve())
print("Rows saved:", len(july_snapshot_df))

In [ ]:
connection.execute(
    """
    CREATE TABLE IF NOT EXISTS refresh_runs (
        refresh_run_id TEXT PRIMARY KEY,
        source_file_name TEXT NOT NULL,
        source_snapshot_date TEXT,
        source_file_hash TEXT NOT NULL,
        started_at TEXT NOT NULL,
        completed_at TEXT,
        run_status TEXT NOT NULL
            CHECK (
                run_status IN (
                    'Started', 'Succeeded', 'Rejected', 'Failed'
                )
            ),
        rows_received INTEGER NOT NULL DEFAULT 0,
        rows_inserted INTEGER NOT NULL DEFAULT 0,
        rows_updated INTEGER NOT NULL DEFAULT 0,
        rows_unchanged INTEGER NOT NULL DEFAULT 0,
        rows_rejected INTEGER NOT NULL DEFAULT 0,
        operational_records_preserved INTEGER NOT NULL DEFAULT 0
            CHECK (operational_records_preserved IN (0, 1)),
        error_details TEXT,
        CHECK (
            run_status <> 'Succeeded'
            OR operational_records_preserved = 1
        )
    );
    """
)

connection.commit()

print("Refresh log table is ready.")

In [ ]:
connection.execute("PRAGMA foreign_keys = ON;")

connection.execute(
    """
    CREATE TABLE IF NOT EXISTS data_quality_exceptions (
        exception_id INTEGER PRIMARY KEY AUTOINCREMENT,
        refresh_run_id TEXT NOT NULL,
        source_row_number INTEGER NOT NULL,
        case_id TEXT,
        rejection_reason TEXT NOT NULL,
        source_row_json TEXT NOT NULL,
        recorded_at TEXT NOT NULL,
        FOREIGN KEY (refresh_run_id)
            REFERENCES refresh_runs (refresh_run_id)
    );
    """
)

connection.commit()

print("Data-quality exception table is ready.")

In [ ]:
import hashlib
from datetime import datetime, timezone
from uuid import uuid4

refresh_run_id = f"REFRESH-{uuid4().hex}"
refresh_started_at = datetime.now(timezone.utc).isoformat()

snapshot_bytes = july_snapshot_path.read_bytes()
snapshot_hash = hashlib.sha256(snapshot_bytes).hexdigest()

with connection:
    connection.execute(
        """
        INSERT INTO refresh_runs (
            refresh_run_id,
            source_file_name,
            source_file_hash,
            started_at,
            run_status
        )
        VALUES (?, ?, ?, ?, 'Started');
        """,
        (
            refresh_run_id,
            july_snapshot_path.name,
            snapshot_hash,
            refresh_started_at,
        ),
    )

print("Refresh run:", refresh_run_id)
print("Status: Started")
print("File:", july_snapshot_path.name)

In [ ]:
from io import BytesIO

try:
    incoming_snapshot_df = pd.read_csv(
        BytesIO(snapshot_bytes),
        dtype="string",
        keep_default_na=False,
    )

    missing_columns = (
        set(source_columns)
        - set(incoming_snapshot_df.columns)
    )

    if missing_columns:
        raise ValueError(
            f"Missing source columns: {sorted(missing_columns)}"
        )

except Exception as exc:
    with connection:
        connection.execute(
            """
            UPDATE refresh_runs
            SET run_status = ?,
                completed_at = ?,
                error_details = ?
            WHERE refresh_run_id = ?;
            """,
            (
                "Rejected" if isinstance(exc, ValueError) else "Failed",
                datetime.now(timezone.utc).isoformat(),
                str(exc),
                refresh_run_id,
            ),
        )
    raise

with connection:
    connection.execute(
        """
        UPDATE refresh_runs
        SET rows_received = ?
        WHERE refresh_run_id = ?;
        """,
        (len(incoming_snapshot_df), refresh_run_id),
    )

print("Rows received:", len(incoming_snapshot_df))
print("Required source columns are present.")

In [ ]:
# Keep the original values for exception reporting.
validated_snapshot_df = incoming_snapshot_df[
    source_columns
].copy()

for column in source_columns:
    validated_snapshot_df[column] = (
        validated_snapshot_df[column].str.strip()
    )

row_errors = {
    row_index: []
    for row_index in validated_snapshot_df.index
}

for column in source_columns:
    blank_mask = validated_snapshot_df[column].eq("")

    for row_index in validated_snapshot_df.index[blank_mask]:
        row_errors[row_index].append(
            f"Missing required value: {column}"
        )

duplicate_mask = (
    validated_snapshot_df["Case ID"].ne("")
    & validated_snapshot_df["Case ID"].duplicated(keep=False)
)

for row_index in validated_snapshot_df.index[duplicate_mask]:
    row_errors[row_index].append(
        "Duplicate Case ID within incoming snapshot"
    )

print(
    "Rows with missing required values:",
    validated_snapshot_df.eq("").any(axis=1).sum(),
)

print("Rows with duplicate Case IDs:", duplicate_mask.sum())
print(
    "Rows with validation errors so far:",
    sum(bool(errors) for errors in row_errors.values()),
)

In [ ]:
from decimal import Decimal, InvalidOperation

parsed_snapshot_dates = pd.to_datetime(
    validated_snapshot_df["Snapshot Date"],
    format="%Y-%m-%d",
    errors="coerce",
)

for row_index, parsed_date in parsed_snapshot_dates.items():
    if pd.isna(parsed_date):
        row_errors[row_index].append("Invalid snapshot date")
    elif not parsed_date.is_month_end:
        row_errors[row_index].append(
            "Snapshot date must be a month-end date"
        )
    elif parsed_date != next_snapshot_date:
        row_errors[row_index].append(
            "Snapshot date does not match the expected reporting month"
        )

validated_amounts = {}

for row_index, amount_text in (
    validated_snapshot_df["Outstanding Amount"].items()
):
    try:
        amount = Decimal(amount_text)

        if not amount.is_finite() or amount <= 0:
            raise ValueError()

        if amount != amount.quantize(Decimal("0.01")):
            raise ValueError()

        validated_amounts[row_index] = amount

    except (InvalidOperation, ValueError):
        row_errors[row_index].append(
            "Outstanding amount must be positive and in whole pennies"
        )

valid_statuses = {"Cancelled", "Void"}

invalid_status_mask = ~validated_snapshot_df[
    "Cancellation Status"
].isin(valid_statuses)

for row_index in validated_snapshot_df.index[invalid_status_mask]:
    row_errors[row_index].append("Invalid cancellation status")

print(
    "Rows passing all checks:",
    sum(not errors for errors in row_errors.values()),
)

print(
    "Rows requiring quarantine:",
    sum(bool(errors) for errors in row_errors.values()),
)

In [ ]:
import json

accepted_indices = [
    row_index
    for row_index, errors in row_errors.items()
    if not errors
]

accepted_snapshot_df = validated_snapshot_df.loc[
    accepted_indices
].copy()

exception_rows = [
    (
        refresh_run_id,
        int(row_index) + 2,
        incoming_snapshot_df.at[row_index, "Case ID"],
        "; ".join(errors),
        json.dumps(
            incoming_snapshot_df.loc[row_index].to_dict()
        ),
        datetime.now(timezone.utc).isoformat(),
    )
    for row_index, errors in row_errors.items()
    if errors
]

with connection:
    connection.executemany(
        """
        INSERT INTO data_quality_exceptions (
            refresh_run_id,
            source_row_number,
            case_id,
            rejection_reason,
            source_row_json,
            recorded_at
        )
        VALUES (?, ?, ?, ?, ?, ?);
        """,
        exception_rows,
    )

    connection.execute(
        """
        UPDATE refresh_runs
        SET source_snapshot_date = ?,
            rows_rejected = ?
        WHERE refresh_run_id = ?;
        """,
        (
            next_snapshot_date.strftime("%Y-%m-%d"),
            len(exception_rows),
            refresh_run_id,
        ),
    )

print("Accepted rows:", len(accepted_snapshot_df))
print("Rejected rows retained:", len(exception_rows))

In [ ]:
source_table_columns = {
    row[1]
    for row in connection.execute(
        "PRAGMA table_info(source_cases);"
    ).fetchall()
}

with connection:
    if "First Identified Date" not in source_table_columns:
        connection.execute(
            """
            ALTER TABLE source_cases
            ADD COLUMN "First Identified Date" TEXT;
            """
        )

    connection.execute(
        """
        UPDATE source_cases
        SET "First Identified Date" = "Snapshot Date"
        WHERE "First Identified Date" IS NULL;
        """
    )

date_check_df = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS source_cases,
        COUNT("First Identified Date") AS original_dates_preserved
    FROM source_cases;
    """,
    connection,
)

date_check_df

In [ ]:
current_source_df = pd.read_sql_query(
    "SELECT * FROM source_cases;",
    connection,
).set_index("Case ID")

refresh_actions = []

for row_index, row in accepted_snapshot_df.iterrows():
    case_id = row["Case ID"]

    if case_id not in current_source_df.index:
        action = "Insert"
    else:
        existing = current_source_df.loc[case_id]

        text_changed = any(
            str(existing[column]) != str(row[column])
            for column in [
                "Policy Number",
                "Client Number",
                "Cancellation Status",
                "Cancelling Department",
                "Cancelling Agent",
            ]
        )

        date_changed = (
            pd.Timestamp(existing["Snapshot Date"]).normalize()
            != parsed_snapshot_dates.loc[row_index]
        )

        amount_changed = (
            Decimal(str(existing["Outstanding Amount"]))
            != validated_amounts[row_index]
        )

        action = (
            "Update"
            if text_changed or date_changed or amount_changed
            else "Unchanged"
        )

    refresh_actions.append({
        "Case ID": case_id,
        "Action": action,
    })

refresh_preview_df = pd.DataFrame(refresh_actions)

print(refresh_preview_df["Action"].value_counts().to_string())

In [ ]:
refresh_time = datetime.now(timezone.utc).isoformat()

source_rows = []

for row_index, row in accepted_snapshot_df.iterrows():
    snapshot_date = parsed_snapshot_dates.loc[row_index].strftime(
        "%Y-%m-%d"
    )

    source_rows.append((
        row["Case ID"],
        row["Policy Number"],
        row["Client Number"],
        snapshot_date,
        float(validated_amounts[row_index]),
        row["Cancellation Status"],
        row["Cancelling Department"],
        row["Cancelling Agent"],
        snapshot_date,
    ))

with connection:
    connection.executemany(
        """
        INSERT INTO source_cases (
            "Case ID", "Policy Number", "Client Number",
            "Snapshot Date", "Outstanding Amount",
            "Cancellation Status", "Cancelling Department",
            "Cancelling Agent", "First Identified Date"
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT ("Case ID") DO UPDATE SET
            "Policy Number" = excluded."Policy Number",
            "Client Number" = excluded."Client Number",
            "Snapshot Date" = excluded."Snapshot Date",
            "Outstanding Amount" = excluded."Outstanding Amount",
            "Cancellation Status" = excluded."Cancellation Status",
            "Cancelling Department" = excluded."Cancelling Department",
            "Cancelling Agent" = excluded."Cancelling Agent";
        """,
        source_rows,
    )

    connection.executemany(
        """
        INSERT INTO agent_updates (
            "Case ID", "Case Status", "Last Updated Date"
        )
        SELECT ?, 'Open', ?
        WHERE NOT EXISTS (
            SELECT 1
            FROM agent_updates
            WHERE "Case ID" = ?
        );
        """,
        [
            (row[0], refresh_time, row[0])
            for row in source_rows
        ],
    )

print("July source changes applied.")
print("Existing operational records were not updated.")

In [ ]:
current_updates_df = pd.read_sql_query(
    "SELECT * FROM agent_updates;",
    connection,
)

saved_baseline_df = pd.read_sql_query(
    "SELECT * FROM baseline_agent_updates;",
    connection,
)

baseline_ids = set(saved_baseline_df["Case ID"])

original_records_now = (
    current_updates_df.loc[
        current_updates_df["Case ID"].isin(baseline_ids),
        required_update_columns,
    ]
    .sort_values("Case ID")
    .reset_index(drop=True)
)

original_records_before = (
    saved_baseline_df[required_update_columns]
    .sort_values("Case ID")
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    original_records_now,
    original_records_before,
)

new_records = current_updates_df.loc[
    ~current_updates_df["Case ID"].isin(baseline_ids)
]

expected_new_ids = set(new_cases_df["Case ID"])

assert set(new_records["Case ID"]) == expected_new_ids
assert new_records["Case Status"].eq("Open").all()

source_count = connection.execute(
    "SELECT COUNT(*) FROM source_cases;"
).fetchone()[0]

assert source_count == 2873
assert len(current_updates_df) == 2873

print("All 2,871 original operational records are unchanged.")
print("Both new cases have Open operational records.")
print("Source cases:", source_count)
print("Agent records:", len(current_updates_df))
print(
    "Completed cases:",
    current_updates_df["Case Status"].eq("Completed").sum(),
)

In [ ]:
saved_source_df = pd.read_sql_query(
    "SELECT * FROM source_cases;",
    connection,
).set_index("Case ID")

for row_index, row in accepted_snapshot_df.iterrows():
    saved = saved_source_df.loc[row["Case ID"]]

    for column in source_columns:
        if column == "Case ID":
            continue
        elif column == "Snapshot Date":
            assert (
                pd.Timestamp(saved[column])
                == parsed_snapshot_dates.loc[row_index]
            )
        elif column == "Outstanding Amount":
            assert (
                Decimal(str(saved[column]))
                == validated_amounts[row_index]
            )
        else:
            assert saved[column] == row[column]

original_dates = baseline_source_df.set_index("Case ID")[
    "Snapshot Date"
]

for case_id, original_date in original_dates.items():
    assert (
        pd.Timestamp(
            saved_source_df.loc[case_id, "First Identified Date"]
        )
        == original_date
    )

for case_id in expected_new_ids:
    assert (
        pd.Timestamp(
            saved_source_df.loc[case_id, "First Identified Date"]
        )
        == next_snapshot_date
    )

print("All 12 accepted source rows match July's input.")
print("All 2,871 original identification dates are unchanged.")
print("Both new cases have July's identification date.")

In [ ]:
action_counts = refresh_preview_df["Action"].value_counts()

with connection:
    cursor = connection.execute(
        """
        UPDATE refresh_runs
        SET completed_at = ?,
            run_status = 'Succeeded',
            rows_inserted = ?,
            rows_updated = ?,
            rows_unchanged = ?,
            operational_records_preserved = 1
        WHERE refresh_run_id = ?
          AND run_status = 'Started';
        """,
        (
            datetime.now(timezone.utc).isoformat(),
            int(action_counts.get("Insert", 0)),
            int(action_counts.get("Update", 0)),
            int(action_counts.get("Unchanged", 0)),
            refresh_run_id,
        ),
    )

    if cursor.rowcount != 1:
        raise ValueError(
            "Expected one Started refresh run to finalise."
        )

pd.read_sql_query(
    """
    SELECT
        run_status,
        rows_received,
        rows_inserted,
        rows_updated,
        rows_unchanged,
        rows_rejected,
        operational_records_preserved
    FROM refresh_runs
    WHERE refresh_run_id = ?;
    """,
    connection,
    params=(refresh_run_id,),
)

In [ ]:
before_repeat_source_df = pd.read_sql_query(
    'SELECT * FROM source_cases ORDER BY "Case ID";',
    connection,
)

before_repeat_updates_df = pd.read_sql_query(
    'SELECT * FROM agent_updates ORDER BY "Case ID";',
    connection,
)

refresh_run_id = f"REFRESH-{uuid4().hex}"

with connection:
    connection.execute(
        """
        INSERT INTO refresh_runs (
            refresh_run_id,
            source_file_name,
            source_snapshot_date,
            source_file_hash,
            started_at,
            run_status,
            rows_received,
            rows_rejected
        )
        VALUES (?, ?, ?, ?, ?, 'Started', ?, 0);
        """,
        (
            refresh_run_id,
            july_snapshot_path.name,
            next_snapshot_date.strftime("%Y-%m-%d"),
            snapshot_hash,
            datetime.now(timezone.utc).isoformat(),
            len(accepted_snapshot_df),
        ),
    )

print("Repeat attempt recorded:", refresh_run_id)
print("Before-repeat copies captured.")

In [ ]:
after_repeat_source_df = pd.read_sql_query(
    'SELECT * FROM source_cases ORDER BY "Case ID";',
    connection,
)

after_repeat_updates_df = pd.read_sql_query(
    'SELECT * FROM agent_updates ORDER BY "Case ID";',
    connection,
)

pd.testing.assert_frame_equal(
    after_repeat_source_df,
    before_repeat_source_df,
)

pd.testing.assert_frame_equal(
    after_repeat_updates_df,
    before_repeat_updates_df,
)

assert refresh_preview_df["Action"].eq("Unchanged").all()
assert len(refresh_preview_df) == 12

print("Source table is unchanged after repeat processing.")
print("Agent records are unchanged after repeat processing.")
print("No duplicate cases or operational records were created.")

In [ ]:
invalid_snapshot_df = july_snapshot_df.copy()

invalid_snapshot_df["Snapshot Date"] = (
    invalid_snapshot_df["Snapshot Date"]
    .dt.strftime("%Y-%m-%d")
)

invalid_snapshot_df = invalid_snapshot_df.astype("string")

# Reject both occurrences of the duplicated Case ID.
invalid_snapshot_df.loc[1, "Case ID"] = (
    invalid_snapshot_df.loc[0, "Case ID"]
)

invalid_snapshot_df.loc[2, "Client Number"] = ""
invalid_snapshot_df.loc[3, "Snapshot Date"] = "not-a-date"
invalid_snapshot_df.loc[4, "Outstanding Amount"] = "-5.00"
invalid_snapshot_df.loc[5, "Cancellation Status"] = "Unknown"

invalid_snapshot_path = (
    test_folder / "snapshot_2026_07_invalid.csv"
)

if invalid_snapshot_path.exists():
    raise FileExistsError(
        "The invalid test file already exists."
    )

invalid_snapshot_df.to_csv(
    invalid_snapshot_path,
    index=False,
)

print("Invalid test file saved.")
print("Total rows:", len(invalid_snapshot_df))
print("Expected accepted rows: 6")
print("Expected rejected rows: 6")

In [ ]:
before_invalid_source_df = pd.read_sql_query(
    'SELECT * FROM source_cases ORDER BY "Case ID";',
    connection,
)

before_invalid_updates_df = pd.read_sql_query(
    'SELECT * FROM agent_updates ORDER BY "Case ID";',
    connection,
)

snapshot_bytes = invalid_snapshot_path.read_bytes()
snapshot_hash = hashlib.sha256(snapshot_bytes).hexdigest()

refresh_run_id = f"REFRESH-{uuid4().hex}"

with connection:
    connection.execute(
        """
        INSERT INTO refresh_runs (
            refresh_run_id,
            source_file_name,
            source_file_hash,
            started_at,
            run_status
        )
        VALUES (?, ?, ?, ?, 'Started');
        """,
        (
            refresh_run_id,
            invalid_snapshot_path.name,
            snapshot_hash,
            datetime.now(timezone.utc).isoformat(),
        ),
    )

print("Invalid-file attempt recorded:", refresh_run_id)
print("File:", invalid_snapshot_path.name)
print("Before-refresh copies captured.")

In [ ]:
rejection_preview_df = pd.DataFrame([
    {
        "CSV Row": int(row_index) + 2,
        "Case ID": incoming_snapshot_df.at[row_index, "Case ID"],
        "Rejection Reason": "; ".join(errors),
    }
    for row_index, errors in row_errors.items()
    if errors
])

rejection_preview_df

In [ ]:
saved_exceptions_df = pd.read_sql_query(
    """
    SELECT source_row_number, case_id, rejection_reason
    FROM data_quality_exceptions
    WHERE refresh_run_id = ?
    ORDER BY source_row_number;
    """,
    connection,
    params=(refresh_run_id,),
)

assert len(saved_exceptions_df) == 6

assert set(accepted_snapshot_df["Case ID"]) == {
    "DH000007",
    "DH000008",
    "DH000030",
    "DH000050",
    "TEST-JUL-001",
    "TEST-JUL-002",
}

print("All 6 rejected rows are retained in quarantine.")
print("Only the 6 valid rows remain eligible for loading.")

In [ ]:
after_invalid_source_df = pd.read_sql_query(
    'SELECT * FROM source_cases ORDER BY "Case ID";',
    connection,
)

after_invalid_updates_df = pd.read_sql_query(
    'SELECT * FROM agent_updates ORDER BY "Case ID";',
    connection,
)

pd.testing.assert_frame_equal(
    after_invalid_source_df,
    before_invalid_source_df,
)

pd.testing.assert_frame_equal(
    after_invalid_updates_df,
    before_invalid_updates_df,
)

assert len(accepted_snapshot_df) == 6
assert len(exception_rows) == 6
assert refresh_preview_df["Action"].eq("Unchanged").all()

print("Both case tables remain unchanged.")
print("Rejected rows did not overwrite existing cases.")
print("No duplicate records were created.")

In [4]:
def preview_source_refresh(accepted_rows, db_connection):
    current_source = pd.read_sql_query(
        "SELECT * FROM source_cases;",
        db_connection,
    ).set_index("Case ID")

    text_columns = [
        "Policy Number",
        "Client Number",
        "Cancellation Status",
        "Cancelling Department",
        "Cancelling Agent",
    ]

    actions = []

    for _, row in accepted_rows.iterrows():
        case_id = row["Case ID"]

        if case_id not in current_source.index:
            action = "Insert"
        else:
            existing = current_source.loc[case_id]

            changed = (
                any(
                    existing[column] != row[column]
                    for column in text_columns
                )
                or pd.Timestamp(existing["Snapshot Date"])
                != pd.Timestamp(row["Snapshot Date"])
                or Decimal(str(existing["Outstanding Amount"]))
                != Decimal(str(row["Outstanding Amount"]))
            )

            action = "Update" if changed else "Unchanged"

        actions.append({"Case ID": case_id, "Action": action})

    return pd.DataFrame(actions, columns=["Case ID", "Action"])

In [ ]:
preview = preview_source_refresh(
    accepted_snapshot_df,
    connection,
)

print(preview["Action"].value_counts().to_string())

In [3]:
def validate_source_snapshot(incoming_rows, expected_snapshot_date):
    missing_columns = set(source_columns) - set(incoming_rows.columns)

    if missing_columns:
        raise ValueError(
            f"Missing source columns: {sorted(missing_columns)}"
        )

    original_rows = incoming_rows[source_columns].copy()
    cleaned_rows = original_rows.astype("string").fillna("")

    for column in source_columns:
        cleaned_rows[column] = cleaned_rows[column].str.strip()

    errors = {index: [] for index in cleaned_rows.index}

    for column in source_columns:
        for index in cleaned_rows.index[cleaned_rows[column].eq("")]:
            errors[index].append(f"Missing required value: {column}")

    duplicates = (
        cleaned_rows["Case ID"].ne("")
        & cleaned_rows["Case ID"].duplicated(keep=False)
    )

    for index in cleaned_rows.index[duplicates]:
        errors[index].append(
            "Duplicate Case ID within incoming snapshot"
        )

    dates = pd.to_datetime(
        cleaned_rows["Snapshot Date"],
        format="%Y-%m-%d",
        errors="coerce",
    )

    for index, date in dates.items():
        if pd.isna(date):
            errors[index].append("Invalid snapshot date")
        elif not date.is_month_end:
            errors[index].append(
                "Snapshot date must be a month-end date"
            )
        elif date != pd.Timestamp(expected_snapshot_date):
            errors[index].append(
                "Snapshot date does not match the expected reporting month"
            )

    for index, value in cleaned_rows["Outstanding Amount"].items():
        try:
            amount = Decimal(value)

            if (
                not amount.is_finite()
                or amount <= 0
                or amount != amount.quantize(Decimal("0.01"))
            ):
                raise ValueError()

        except (InvalidOperation, ValueError):
            errors[index].append(
                "Outstanding amount must be positive and in whole pennies"
            )

    invalid_status = ~cleaned_rows["Cancellation Status"].isin(
        {"Cancelled", "Void"}
    )

    for index in cleaned_rows.index[invalid_status]:
        errors[index].append("Invalid cancellation status")

    accepted = cleaned_rows.loc[
        [index for index, reasons in errors.items() if not reasons]
    ].copy()

    rejected = pd.DataFrame(
        [
            {
                "source_row_number": int(index) + 2,
                "case_id": original_rows.at[index, "Case ID"],
                "rejection_reason": "; ".join(reasons),
                "source_row_json": json.dumps(
                    original_rows.loc[index].to_dict()
                ),
            }
            for index, reasons in errors.items()
            if reasons
        ],
        columns=[
            "source_row_number",
            "case_id",
            "rejection_reason",
            "source_row_json",
        ],
    )

    return accepted, rejected

In [ ]:
accepted_test, rejected_test = validate_source_snapshot(
    incoming_snapshot_df,
    next_snapshot_date,
)

assert len(accepted_test) == 6
assert len(rejected_test) == 6

print("Accepted:", len(accepted_test))
print("Rejected:", len(rejected_test))

In [5]:
def apply_source_refresh(accepted_rows, db_connection, updated_at):
    source_rows = []

    for _, row in accepted_rows.iterrows():
        snapshot_date = pd.Timestamp(
            row["Snapshot Date"]
        ).strftime("%Y-%m-%d")

        source_rows.append((
            row["Case ID"],
            row["Policy Number"],
            row["Client Number"],
            snapshot_date,
            float(Decimal(str(row["Outstanding Amount"]))),
            row["Cancellation Status"],
            row["Cancelling Department"],
            row["Cancelling Agent"],
            snapshot_date,
        ))

    db_connection.executemany(
        """
        INSERT INTO source_cases (
            "Case ID", "Policy Number", "Client Number",
            "Snapshot Date", "Outstanding Amount",
            "Cancellation Status", "Cancelling Department",
            "Cancelling Agent", "First Identified Date"
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT ("Case ID") DO UPDATE SET
            "Policy Number" = excluded."Policy Number",
            "Client Number" = excluded."Client Number",
            "Snapshot Date" = excluded."Snapshot Date",
            "Outstanding Amount" = excluded."Outstanding Amount",
            "Cancellation Status" = excluded."Cancellation Status",
            "Cancelling Department" = excluded."Cancelling Department",
            "Cancelling Agent" = excluded."Cancelling Agent";
        """,
        source_rows,
    )

    db_connection.executemany(
        """
        INSERT INTO agent_updates (
            "Case ID", "Case Status", "Last Updated Date"
        )
        SELECT ?, 'Open', ?
        WHERE NOT EXISTS (
            SELECT 1 FROM agent_updates
            WHERE "Case ID" = ?
        );
        """,
        [
            (row[0], updated_at, row[0])
            for row in source_rows
        ],
    )

In [6]:
def verify_operational_preservation(
    before_updates,
    db_connection,
    expected_new_ids,
):
    after_updates = pd.read_sql_query(
        "SELECT * FROM agent_updates;",
        db_connection,
    )

    if after_updates["Case ID"].duplicated().any():
        raise ValueError("Duplicate operational Case IDs detected.")

    before_ids = set(before_updates["Case ID"])
    after_ids = set(after_updates["Case ID"])

    if after_ids != before_ids | set(expected_new_ids):
        raise ValueError("Unexpected operational records added or removed.")

    original_records_after = (
        after_updates.loc[
            after_updates["Case ID"].isin(before_ids)
        ]
        .sort_values("Case ID")
        .reset_index(drop=True)
    )

    original_records_before = (
        before_updates
        .sort_values("Case ID")
        .reset_index(drop=True)
    )

    pd.testing.assert_frame_equal(
        original_records_after,
        original_records_before,
    )

    new_records = after_updates.loc[
        after_updates["Case ID"].isin(expected_new_ids)
    ]

    if not new_records["Case Status"].eq("Open").all():
        raise ValueError("New operational records must be Open.")

    blank_columns = [
        "Agent Working",
        "Agent Notes",
        "Final Outcome",
        "Completed By",
        "Completion Date",
    ]

    if not new_records[blank_columns].isna().all().all():
        raise ValueError("New operational records contain unexpected values.")

    if new_records["Last Updated Date"].isna().any():
        raise ValueError("New operational records need an update timestamp.")

In [8]:
def run_monthly_refresh(file_path, expected_date, db_connection):
    if db_connection.in_transaction:
        raise RuntimeError("Finish the existing transaction first.")

    file_path = Path(file_path)
    run_id = f"REFRESH-{uuid4().hex}"
    started_at = datetime.now(timezone.utc).isoformat()

    file_bytes = file_path.read_bytes()
    file_hash = hashlib.sha256(file_bytes).hexdigest()

    with db_connection:
        db_connection.execute(
            """
            INSERT INTO refresh_runs (
                refresh_run_id,
                source_file_name,
                source_file_hash,
                started_at,
                run_status
            )
            VALUES (?, ?, ?, ?, 'Started');
            """,
            (run_id, file_path.name, file_hash, started_at),
        )

    stage = "read"

    try:
        incoming = pd.read_csv(
            BytesIO(file_bytes),
            dtype="string",
            keep_default_na=False,
        )

        with db_connection:
            db_connection.execute(
                """
                UPDATE refresh_runs
                SET rows_received = ?
                WHERE refresh_run_id = ?;
                """,
                (len(incoming), run_id),
            )

        stage = "validation"

        accepted, rejected = validate_source_snapshot(
            incoming,
            expected_date,
        )

        with db_connection:
            db_connection.executemany(
                """
                INSERT INTO data_quality_exceptions (
                    refresh_run_id,
                    source_row_number,
                    case_id,
                    rejection_reason,
                    source_row_json,
                    recorded_at
                )
                VALUES (?, ?, ?, ?, ?, ?);
                """,
                [
                    (
                        run_id,
                        int(row["source_row_number"]),
                        row["case_id"],
                        row["rejection_reason"],
                        row["source_row_json"],
                        started_at,
                    )
                    for _, row in rejected.iterrows()
                ],
            )

            db_connection.execute(
                """
                UPDATE refresh_runs
                SET source_snapshot_date = ?,
                    rows_rejected = ?
                WHERE refresh_run_id = ?;
                """,
                (
                    pd.Timestamp(expected_date).strftime("%Y-%m-%d"),
                    len(rejected),
                    run_id,
                ),
            )

        if accepted.empty:
            raise ValueError("No valid source rows remain.")

        db_connection.execute("BEGIN IMMEDIATE;")

        # Check against the latest saved source data while locked.
        check_snapshot_order(db_connection, expected_date)

        stage = "apply"

        before_updates = pd.read_sql_query(
            "SELECT * FROM agent_updates;",
            db_connection,
        )

        preview = preview_source_refresh(accepted, db_connection)

        expected_new_ids = (
            set(accepted["Case ID"])
            - set(before_updates["Case ID"])
        )

        apply_source_refresh(
            accepted,
            db_connection,
            started_at,
        )

        verify_operational_preservation(
            before_updates,
            db_connection,
            expected_new_ids,
        )

        counts = preview["Action"].value_counts()

        if sum(counts) + len(rejected) != len(incoming):
            raise ValueError("Refresh row counts do not reconcile.")

        db_connection.execute(
            """
            UPDATE refresh_runs
            SET completed_at = ?,
                run_status = 'Succeeded',
                rows_inserted = ?,
                rows_updated = ?,
                rows_unchanged = ?,
                operational_records_preserved = 1
            WHERE refresh_run_id = ?;
            """,
            (
                datetime.now(timezone.utc).isoformat(),
                int(counts.get("Insert", 0)),
                int(counts.get("Update", 0)),
                int(counts.get("Unchanged", 0)),
                run_id,
            ),
        )

        db_connection.commit()

    except Exception as exc:
        db_connection.rollback()

        status = (
            "Rejected"
            if stage == "validation" and isinstance(exc, ValueError)
            else "Failed"
        )

        with db_connection:
            db_connection.execute(
                """
                UPDATE refresh_runs
                SET completed_at = ?,
                    run_status = ?,
                    error_details = ?
                WHERE refresh_run_id = ?;
                """,
                (
                    datetime.now(timezone.utc).isoformat(),
                    status,
                    str(exc),
                    run_id,
                ),
            )

        raise

    return pd.read_sql_query(
        """
        SELECT
            refresh_run_id,
            run_status,
            rows_received,
            rows_inserted,
            rows_updated,
            rows_unchanged,
            rows_rejected,
            operational_records_preserved
        FROM refresh_runs
        WHERE refresh_run_id = ?;
        """,
        db_connection,
        params=(run_id,),
    )

In [ ]:
valid_refresh_result = run_monthly_refresh(
    july_snapshot_path,
    next_snapshot_date,
    connection,
)

valid_refresh_result

In [ ]:
invalid_refresh_result = run_monthly_refresh(
    invalid_snapshot_path,
    next_snapshot_date,
    connection,
)

invalid_refresh_result

In [ ]:
invalid_run_id = invalid_refresh_result.loc[0, "refresh_run_id"]

exception_count = connection.execute(
    """
    SELECT COUNT(*)
    FROM data_quality_exceptions
    WHERE refresh_run_id = ?;
    """,
    (invalid_run_id,),
).fetchone()[0]

assert exception_count == 6

print("Rejected rows retained for this run:", exception_count)

In [ ]:
missing_column_path = test_folder / "snapshot_missing_column.csv"

july_snapshot_df.drop(
    columns=["Client Number"]
).to_csv(
    missing_column_path,
    index=False,
    date_format="%Y-%m-%d",
)

before_failure_source = pd.read_sql_query(
    'SELECT * FROM source_cases ORDER BY "Case ID";',
    connection,
)

before_failure_updates = pd.read_sql_query(
    'SELECT * FROM agent_updates ORDER BY "Case ID";',
    connection,
)

try:
    run_monthly_refresh(
        missing_column_path,
        next_snapshot_date,
        connection,
    )
except ValueError as exc:
    print("Expected rejection:", exc)
else:
    raise AssertionError("The invalid file was not rejected.")

pd.testing.assert_frame_equal(
    before_failure_source,
    pd.read_sql_query(
        'SELECT * FROM source_cases ORDER BY "Case ID";',
        connection,
    ),
)

pd.testing.assert_frame_equal(
    before_failure_updates,
    pd.read_sql_query(
        'SELECT * FROM agent_updates ORDER BY "Case ID";',
        connection,
    ),
)

print("Both case tables remain unchanged.")

pd.read_sql_query(
    """
    SELECT run_status, rows_received, error_details
    FROM refresh_runs
    WHERE source_file_name = ?
    ORDER BY started_at DESC
    LIMIT 1;
    """,
    connection,
    params=(missing_column_path.name,),
)

In [ ]:
rollback_test_path = test_folder / "snapshot_rollback_test.csv"

rollback_input = july_snapshot_df.copy()
rollback_input.loc[
    rollback_input["Case ID"].eq("DH000001"),
    "Outstanding Amount",
] = 9.99

rollback_input.to_csv(
    rollback_test_path,
    index=False,
    date_format="%Y-%m-%d",
)

before_rollback_source = pd.read_sql_query(
    'SELECT * FROM source_cases ORDER BY "Case ID";',
    connection,
)

before_rollback_updates = pd.read_sql_query(
    'SELECT * FROM agent_updates ORDER BY "Case ID";',
    connection,
)

try:
    connection.execute(
        """
        CREATE TEMP TRIGGER force_refresh_test_failure
        BEFORE UPDATE ON main.source_cases
        WHEN NEW."Case ID" = 'DH000002'
        BEGIN
            SELECT RAISE(ABORT, 'Intentional rollback test');
        END;
        """
    )
    connection.commit()

    try:
        run_monthly_refresh(
            rollback_test_path,
            next_snapshot_date,
            connection,
        )
    except sqlite3.IntegrityError as exc:
        if "Intentional rollback test" not in str(exc):
            raise
        print("Expected processing failure:", exc)
    else:
        raise AssertionError("The intentional failure did not occur.")

finally:
    connection.execute(
        "DROP TRIGGER IF EXISTS temp.force_refresh_test_failure;"
    )
    connection.commit()

pd.testing.assert_frame_equal(
    before_rollback_source,
    pd.read_sql_query(
        'SELECT * FROM source_cases ORDER BY "Case ID";',
        connection,
    ),
)

pd.testing.assert_frame_equal(
    before_rollback_updates,
    pd.read_sql_query(
        'SELECT * FROM agent_updates ORDER BY "Case ID";',
        connection,
    ),
)

print("All case changes rolled back successfully.")

pd.read_sql_query(
    """
    SELECT run_status, error_details
    FROM refresh_runs
    WHERE source_file_name = ?
    ORDER BY started_at DESC
    LIMIT 1;
    """,
    connection,
    params=(rollback_test_path.name,),
)

In [ ]:
fresh_test_path = test_folder / "monthly_refresh_fresh_test.db"

if fresh_test_path.exists():
    raise FileExistsError(
        "Fresh test database already exists. Stop to avoid reusing it."
    )

fresh_connection = sqlite3.connect(fresh_test_path)
fresh_connection.execute("PRAGMA foreign_keys = ON;")

# Copy table definitions, not the July data.
for table_name in [
    "source_cases",
    "agent_updates",
    "refresh_runs",
    "data_quality_exceptions",
]:
    table_sql = connection.execute(
        """
        SELECT sql
        FROM sqlite_master
        WHERE type = 'table' AND name = ?;
        """,
        (table_name,),
    ).fetchone()[0]

    fresh_connection.execute(table_sql)

# Copy the explicit unique indexes for case identifiers.
for index_name in [
    "source_case_id_unique",
    "agent_case_id_unique",
]:
    index_sql = connection.execute(
        """
        SELECT sql
        FROM sqlite_master
        WHERE type = 'index' AND name = ?;
        """,
        (index_name,),
    ).fetchone()[0]

    fresh_connection.execute(index_sql)

fresh_source = baseline_source_df.copy()
fresh_source["Snapshot Date"] = (
    fresh_source["Snapshot Date"].dt.strftime("%Y-%m-%d")
)
fresh_source["First Identified Date"] = fresh_source["Snapshot Date"]

fresh_updates = pd.read_sql_query(
    "SELECT * FROM baseline_agent_updates;",
    connection,
)

for table_name, frame in [
    ("source_cases", fresh_source),
    ("agent_updates", fresh_updates),
]:
    columns = ", ".join(f'"{column}"' for column in frame.columns)
    placeholders = ", ".join("?" for _ in frame.columns)

    clean_frame = frame.astype("object").where(frame.notna(), None)

    fresh_connection.executemany(
        f'INSERT INTO "{table_name}" ({columns}) '
        f"VALUES ({placeholders});",
        clean_frame.itertuples(index=False, name=None),
    )

fresh_connection.commit()

print("Fresh baseline source cases:", len(fresh_source))
print("Fresh baseline agent records:", len(fresh_updates))

In [ ]:
fresh_result = run_monthly_refresh(
    july_snapshot_path,
    next_snapshot_date,
    fresh_connection,
)

fresh_result

In [ ]:
fresh_saved_source = pd.read_sql_query(
    "SELECT * FROM source_cases;",
    fresh_connection,
).set_index("Case ID")

# Build the complete expected source table.
expected_source = fresh_source.set_index("Case ID").copy()

for _, row in july_snapshot_df.iterrows():
    case_id = row["Case ID"]

    if case_id not in expected_source.index:
        expected_source.loc[case_id, "First Identified Date"] = (
            pd.Timestamp(row["Snapshot Date"]).strftime("%Y-%m-%d")
        )

    for column in source_columns:
        if column == "Case ID":
            continue

        value = row[column]

        if column == "Snapshot Date":
            value = pd.Timestamp(value).strftime("%Y-%m-%d")

        expected_source.loc[case_id, column] = value

actual_comparison = (
    fresh_saved_source
    .sort_index()
    .reset_index()
)

expected_comparison = (
    expected_source[fresh_saved_source.columns]
    .sort_index()
    .reset_index()
)

pd.testing.assert_frame_equal(
    actual_comparison,
    expected_comparison,
    check_dtype=False,
    check_exact=True,
)

verify_operational_preservation(
    fresh_updates,
    fresh_connection,
    set(new_cases_df["Case ID"]),
)

assert len(fresh_saved_source) == 2873

print("All source values match the expected refresh.")
print("Original identification dates are preserved.")
print("Cases absent from July's file are retained.")
print("Existing operational records are unchanged.")
print("Both new cases have default Open records.")
print("Total source cases:", len(fresh_saved_source))

In [7]:
def check_snapshot_order(db_connection, incoming_snapshot_date):
    incoming_date = pd.Timestamp(incoming_snapshot_date)

    if pd.isna(incoming_date) or not incoming_date.is_month_end:
        raise ValueError("Incoming snapshot must have a month-end date.")

    latest_saved_date = db_connection.execute(
        """
        SELECT MAX(date("Snapshot Date"))
        FROM source_cases;
        """
    ).fetchone()[0]

    if latest_saved_date is None:
        return

    latest_saved_date = pd.Timestamp(latest_saved_date)

    if incoming_date < latest_saved_date:
        raise ValueError(
            f"Out-of-order snapshot: {incoming_date.date()} "
            f"is older than saved source data dated "
            f"{latest_saved_date.date()}."
        )

In [ ]:
check_snapshot_order(connection, pd.Timestamp("2026-07-31"))
print("Same-month July snapshot: allowed.")

try:
    check_snapshot_order(connection, pd.Timestamp("2026-06-30"))
except ValueError as exc:
    print("Older June snapshot: rejected.")
    print(exc)
else:
    raise AssertionError("The older snapshot was not rejected.")

In [ ]:
import hashlib
import json

from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation
from io import BytesIO
from uuid import uuid4

source_columns = [
    "Case ID",
    "Policy Number",
    "Client Number",
    "Snapshot Date",
    "Outstanding Amount",
    "Cancellation Status",
    "Cancelling Department",
    "Cancelling Agent",
]

test_folder = database_path.parent

required_helpers = [
    "validate_source_snapshot",
    "preview_source_refresh",
    "apply_source_refresh",
    "verify_operational_preservation",
    "check_snapshot_order",
]

missing_helpers = [
    name
    for name in required_helpers
    if not callable(globals().get(name))
]

print("Missing helper definitions:", missing_helpers)

In [ ]:
older_snapshot_path = test_folder / "snapshot_2026_06_out_of_order.csv"

older_snapshot = pd.read_csv(
    test_folder / "snapshot_2026_07_valid.csv",
    dtype="string",
    keep_default_na=False,
)

older_snapshot["Snapshot Date"] = "2026-06-30"
older_snapshot.to_csv(older_snapshot_path, index=False)

before_order_test = {
    table: pd.read_sql_query(
        f'SELECT * FROM "{table}" ORDER BY "Case ID";',
        connection,
    )
    for table in ["source_cases", "agent_updates"]
}

existing_run_ids = {
    row[0]
    for row in connection.execute(
        "SELECT refresh_run_id FROM refresh_runs;"
    ).fetchall()
}

try:
    run_monthly_refresh(
        older_snapshot_path,
        pd.Timestamp("2026-06-30"),
        connection,
    )
except ValueError as exc:
    assert "Out-of-order snapshot" in str(exc)
    print("Expected rejection:", exc)
else:
    raise AssertionError("The older snapshot was not rejected.")

for table, expected in before_order_test.items():
    pd.testing.assert_frame_equal(
        expected,
        pd.read_sql_query(
            f'SELECT * FROM "{table}" ORDER BY "Case ID";',
            connection,
        ),
    )

all_runs = pd.read_sql_query(
    """
    SELECT refresh_run_id, run_status, error_details
    FROM refresh_runs;
    """,
    connection,
)

new_run = all_runs.loc[
    ~all_runs["refresh_run_id"].isin(existing_run_ids)
]

assert len(new_run) == 1
assert new_run.iloc[0]["run_status"] == "Rejected"
assert "Out-of-order snapshot" in new_run.iloc[0]["error_details"]

print("Older snapshot rejected and logged.")
print("Source cases and operational records are unchanged.")

In [1]:
print("Notebook 11 works")

Notebook 11 works


In [2]:
from pathlib import Path
from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation
from io import BytesIO
from uuid import uuid4

import hashlib
import json
import sqlite3
import pandas as pd

database_path = Path(
    "../data/prototype/refresh_testing/monthly_refresh_test.db"
).resolve()

if not database_path.is_file():
    raise FileNotFoundError(database_path)

connection = sqlite3.connect(
    database_path.as_uri() + "?mode=rw",
    uri=True,
)

connection.execute("PRAGMA foreign_keys = ON;")

test_folder = database_path.parent

source_columns = [
    "Case ID",
    "Policy Number",
    "Client Number",
    "Snapshot Date",
    "Outstanding Amount",
    "Cancellation Status",
    "Cancelling Department",
    "Cancelling Agent",
]

print("Connected to:", database_path)

Connected to: C:\Missed Refund Detection and Process Improvement\data\prototype\refresh_testing\monthly_refresh_test.db


In [9]:
required_functions = [
    "validate_source_snapshot",
    "preview_source_refresh",
    "apply_source_refresh",
    "verify_operational_preservation",
    "check_snapshot_order",
    "run_monthly_refresh",
]

missing = [
    name
    for name in required_functions
    if not callable(globals().get(name))
]

print("Missing functions:", missing)

Missing functions: []


In [10]:
july_repeat_result = run_monthly_refresh(
    test_folder / "snapshot_2026_07_valid.csv",
    pd.Timestamp("2026-07-31"),
    connection,
)

expected = {
    "run_status": "Succeeded",
    "rows_received": 12,
    "rows_inserted": 0,
    "rows_updated": 0,
    "rows_unchanged": 12,
    "rows_rejected": 0,
    "operational_records_preserved": 1,
}

for column, value in expected.items():
    assert july_repeat_result.loc[0, column] == value, column

print("July repeat passed: 12 unchanged, no inserts or updates.")
print("Operational records preserved.")

july_repeat_result

July repeat passed: 12 unchanged, no inserts or updates.
Operational records preserved.


,refresh_run_id,run_status,rows_received,rows_inserted,rows_updated,rows_unchanged,rows_rejected,operational_records_preserved
0,REFRESH-18dfd37be3ea4c7fa523bc32af2517cf,Succeeded,12,0,0,12,0,1


In [2]:
print("Notebook 11 works")

Notebook 11 works
